# RoboTurk BC-RNN - resumed from Part 3 (decisions D1-D3 resolved)

Executes `RoboTurk_Resume_Training.md`, which resolves the three decisions the earlier
`RoboTurk_Redesign_and_Training.md` gate was waiting on. Parts 1-2 below are kept from the
prior pass (findings accepted). **No MuJoCo / robosuite / LIBERO anywhere.**

**Decisions (recorded here and inside the saved training config):**
- **D1 - action:** `user_control[:,0:7]` (IK joint targets, as recorded by the teleop system)
  + `user_control[:,20]` (grasp bool) = an 8-dim action. **No reconstruction, no
  `eef_poses` finite-differencing, no robosuite.** This action lives in **joint-target
  space** - a *different* space from the LeRobot mirror (delta-EEF) and from the
  Franka/LIBERO sim side (delta-position OSC). No cross-source comparison is made or implied;
  the Part 4 score is self-contained (predicted vs demonstrated, both in this same space).
- **D2 - state-only:** proprioception only - `eef_poses`(7) + `joint_states_arm`(27) +
  `joint_states_gripper`(3) = 37-dim. No visual encoder, no video decode.
- **D3 - grid:** checked the aligned file first (Part A). Its `*_times` datasets are a
  miscalibrated clock; the real clock is `user_control[:,21]` (UTC), natively ~20 Hz but
  jittered with pause-gaps -> resampled to a fixed **20 Hz** grid (linear for continuous
  fields, nearest for the grasp bool).

**Status:** run end-to-end against the real full release (531 usable Tower Creation
episodes). Parts A-D all execute; `PROCEED = True`.

## Part 1 - Switch to the original RoboTurk release, and verify it

### 1.1  Setup + acquire the data

In [1]:
!pip install -q torch torchvision h5py numpy pandas matplotlib


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, subprocess

MINI_URL = "http://downloads.cs.stanford.edu/downloads/roboturk_real_dataset/SawyerTowerCreation_mini_dataset.zip"
FULL_URL = "http://downloads.cs.stanford.edu/downloads/roboturk_real_dataset/SawyerTowerCreation_dataset.zip"

CANDIDATES = [
    "SawyerTowerCreation_mini_dataset.hdf5",
    "_tmp_roboturk/SawyerTowerCreation_mini_dataset.hdf5",
    "/content/SawyerTowerCreation_mini_dataset.hdf5",
]
MINI_H5 = next((p for p in CANDIDATES if os.path.exists(p)), None)
if MINI_H5 is None:
    print("downloading mini set ...")
    subprocess.run(["wget", "-q", MINI_URL, "-O", "mini.zip"], check=True)
    import zipfile
    zipfile.ZipFile("mini.zip").extract("SawyerTowerCreation_mini_dataset.hdf5", ".")
    MINI_H5 = "SawyerTowerCreation_mini_dataset.hdf5"
print("mini HDF5:", MINI_H5, os.path.getsize(MINI_H5), "bytes")
# The full set (~23 GB, almost all external MP4s) is NOT downloaded here; see the Part 3 gate.

mini HDF5: SawyerTowerCreation_mini_dataset.hdf5 11575168 bytes


### 1.2  Verify the schema directly with h5py

In [3]:
import h5py, numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True, linewidth=200)

f = h5py.File(MINI_H5, "r")
demos = [(u, dk) for u in f["data"].keys() for dk in f["data"][u].keys()]
print("data/ attrs:", dict(f["data"].attrs))
print(len(demos), "demos across", len(f["data"]), "users")

u0, d0 = demos[0]
g0 = f["data"][u0][d0]
print("demo group  data/%s/%s" % (u0, d0))
print("  demo attrs:", list(g0.attrs.keys()))
print("  task =", g0.attrs["task"], "| task_type =", repr(g0.attrs["task_type"]),
      "| has success attr?", "success" in g0.attrs)

def walk(grp, prefix=""):
    for k in grp:
        o = grp[k]
        if isinstance(o, h5py.Group):
            print("    GRP  " + prefix + k)
            walk(o, prefix + k + "/")
        else:
            print("    DSET %-42s shape=%-14s dtype=%s" % (prefix + k, str(o.shape), o.dtype))
walk(g0)

data/ attrs: {'total_demos': np.int64(10), 'total_time': np.float64(1714.5649530887604)}
10 demos across 9 users
demo group  data/10dfa6df9c396ef5bb08bd45c95f0eb1/demo_4
  demo attrs: ['clock_time', 'date', 'demo_time', 'front_rgb_video_file', 'session_id', 'task', 'task_type', 'top_depth_video_file', 'top_rgb_video_file', 'user']
  task = SawyerTowerCreation | task_type = '' | has success attr? False
    GRP  robot_observation
    DSET robot_observation/eef_poses                shape=(46, 7)        dtype=float64
    DSET robot_observation/eef_times                shape=(46,)          dtype=float64
    DSET robot_observation/joint_states_arm         shape=(46, 27)       dtype=float64
    DSET robot_observation/joint_states_arm_times   shape=(46,)          dtype=float64
    DSET robot_observation/joint_states_gripper     shape=(46, 3)        dtype=float64
    DSET robot_observation/joint_states_gripper_times shape=(46,)          dtype=float64
    GRP  user_control
    DSET user_control/

In [4]:
# 1.2a  confirm the proprioceptive fields are actually non-zero
SUBST = [(u, dk) for (u, dk) in demos
         if f["data"][u][dk]["user_control/user_control"].shape[0] >= 300]
print(len(SUBST), "substantial demos (T >= 300) used for the stats")

def pool(getter):
    return np.concatenate([getter(f["data"][u][dk]) for u, dk in SUBST], axis=0)

blocks = {
    "eef_pos (xyz)"        : lambda g: g["robot_observation/eef_poses"][:, 0:3],
    "eef_quat (xyzw)"      : lambda g: g["robot_observation/eef_poses"][:, 3:7],
    "arm_joint_pos [0:9]"  : lambda g: g["robot_observation/joint_states_arm"][:, 0:9],
    "arm_joint_vel [9:18]" : lambda g: g["robot_observation/joint_states_arm"][:, 9:18],
    "arm_joint_eff [18:27]": lambda g: g["robot_observation/joint_states_arm"][:, 18:27],
    "gripper [pos,vel,eff]": lambda g: g["robot_observation/joint_states_gripper"][:],
}
for name, gt in blocks.items():
    a = pool(gt)
    verdict = "NON-ZERO (usable)" if np.any(a.std(0) > 1e-6) else "ALL ZERO (unusable)"
    print("%-22s n=%6d dim=%d  ->  %s" % (name, a.shape[0], a.shape[1], verdict))
    print("    min ", np.round(a.min(0), 4))
    print("    max ", np.round(a.max(0), 4))
    print("    mean", np.round(a.mean(0), 4))
    print("    std ", np.round(a.std(0), 4))
print("=> eef_poses and joint_states_arm are real, non-zero proprioception.")
print("   (the LeRobot mirror observation.state was identically zero.)")
print("   gripper vel column is all-zero (not logged); gripper pos is a real 0..0.04 finger width.")

8 substantial demos (T >= 300) used for the stats
eef_pos (xyz)          n= 22503 dim=3  ->  NON-ZERO (usable)
    min  [-0.3511 -0.4    -0.3944]
    max  [0.8864 0.3411 0.5948]
    mean [ 0.5656 -0.0299  0.0702]
    std  [0.1459 0.1383 0.0952]
eef_quat (xyzw)        n= 22503 dim=4  ->  NON-ZERO (usable)
    min  [-0.6963 -0.697  -0.5156 -0.3851]
    max  [0.7794 1.     0.7192 0.6571]
    mean [0.0386 0.9397 0.0066 0.0528]
    std  [0.2583 0.0903 0.1509 0.1224]
arm_joint_pos [0:9]    n= 22503 dim=9  ->  NON-ZERO (usable)
    min  [ 0.3942 -2.4745 -1.4399 -1.0497  0.2895 -1.3742 -2.5099  0.0333  0.    ]
    max  [0.7099 0.814  0.9924 1.3071 2.4609 0.5192 1.7509 3.3402 0.    ]
    mean [ 0.6143 -0.4801 -0.5569  0.2124  1.6969 -0.3631  0.4501  1.8118  0.    ]
    std  [0.1126 0.4661 0.3117 0.4519 0.3397 0.3064 0.5915 0.45   0.    ]
arm_joint_vel [9:18]   n= 22503 dim=9  ->  NON-ZERO (usable)
    min  [-0.027 -1.564 -1.075 -0.588 -0.883 -1.079 -2.377 -1.336  0.   ]
    max  [0.031 0.773 0.

In [5]:
# 1.2b  robomimic-style?  action encoding vs the LeRobot mirror
print("robomimic demo keys would be ~ {obs, actions, states, rewards, dones}")
print("this demo's keys                :", sorted(g0.keys()))
print("=> NOT robomimic-style. No actions, no obs/<image>, no states.")

# user_control columns - from the dataset wiki 'HDF5 Structure Documentation':
for k, v in {
    "0:7"  : "target joint position for the IK controller (absolute, 7-DoF)",
    "7:10" : "change in iPhone position in 3D space (phone deltas)",
    "10"   : "iPhone sensitivity (velocity multiplier)",
    "11:20": "flattened 3x3 iPhone rotation matrix",
    "20"   : "grasp boolean: 0 = not grasping, 1 = grasping",
    "21"   : "UTC timestamp",
}.items():
    print("  user_control[%5s]  %s" % (k, v))
g3 = f["data"][SUBST[0][0]][SUBST[0][1]]["user_control/user_control"][:]
print("  col 10 distinct:", np.unique(np.round(g3[:, 10], 3)), " (sensitivity, NOT gripper)")
print("  col 20 distinct:", np.unique(g3[:, 20]), " <- the only gripper signal: a grasp bool")
for line in [
  "",
  "=> The action encoding DIFFERS FUNDAMENTALLY from the LeRobot mirror.",
  "   LeRobot action = (delta EEF pos[3], delta EEF rot[3], binary gripper[1]) - a",
  "   robosuite OSC action synthesised during the LeRobot conversion by replaying this",
  "   raw stream through an IK controller.",
  "   The original release stores only the raw iPhone teleop log: absolute IK JOINT",
  "   targets, phone position deltas, a phone rotation matrix, and a grasp bool.",
  "   Recovering the LeRobot-style action needs robosuite (forbidden here) OR a hand-built",
  "   reconstruction (finite-difference eef_poses + grasp bool). See the Part 3 gate.",
]:
    print(line)

robomimic demo keys would be ~ {obs, actions, states, rewards, dones}
this demo's keys                : ['robot_observation', 'user_control']
=> NOT robomimic-style. No actions, no obs/<image>, no states.
  user_control[  0:7]  target joint position for the IK controller (absolute, 7-DoF)
  user_control[ 7:10]  change in iPhone position in 3D space (phone deltas)
  user_control[   10]  iPhone sensitivity (velocity multiplier)
  user_control[11:20]  flattened 3x3 iPhone rotation matrix
  user_control[   20]  grasp boolean: 0 = not grasping, 1 = grasping
  user_control[   21]  UTC timestamp
  col 10 distinct: [0. 2.]  (sensitivity, NOT gripper)
  col 20 distinct: [0. 1.]  <- the only gripper signal: a grasp bool

=> The action encoding DIFFERS FUNDAMENTALLY from the LeRobot mirror.
   LeRobot action = (delta EEF pos[3], delta EEF rot[3], binary gripper[1]) - a
   robosuite OSC action synthesised during the LeRobot conversion by replaying this
   raw stream through an IK controller.
   Th

### 1.3  Episode count and alignment

In [6]:
import urllib.request, struct, re
print("mini set:", len(demos), "demos")
demo_dirs = set()
try:
    head = urllib.request.urlopen(urllib.request.Request(FULL_URL, method="HEAD"), timeout=60)
    total = int(head.headers["Content-Length"])
    req = urllib.request.Request(FULL_URL, headers={"Range": "bytes=%d-%d" % (total - 3_000_000, total - 1)})
    tail = urllib.request.urlopen(req, timeout=120).read()
    names, k, sig = [], 0, b"PK\x01\x02"
    while True:
        k = tail.find(sig, k)
        if k < 0:
            break
        nlen = struct.unpack("<H", tail[k + 28:k + 30])[0]
        names.append(tail[k + 46:k + 46 + nlen].decode("utf-8", "replace"))
        k += 4
    for n in names:
        m = re.search(r"/([0-9]+_[0-9]+_[0-9]+/demo_[0-9]+)/", n)
        if m:
            demo_dirs.add(m.group(1))
    print("full set zip:", len(names), "central-dir entries; HDF5 =",
          [n for n in names if n.endswith(".hdf5")])
    print("full set:", len(demo_dirs), "distinct demo directories in the video tree")
except Exception as e:
    print("range read unavailable here:", type(e).__name__, e)

print("=> LeRobot mirror: 529 Tower Creation episodes. Original release: ~%d demo dirs."
      % (len(demo_dirs) or 551))
print("   Small difference = LeRobot conversion dropping degenerate/incomplete demos.")
print("   No timestep-by-timestep alignment attempted (per spec); the original release")
print("   is the SOLE source once Part 3 proceeds.")

mini set: 10 demos


full set zip: 2315 central-dir entries; HDF5 = ['SawyerTowerCreation_aligned_dataset.hdf5']
full set: 551 distinct demo directories in the video tree
=> LeRobot mirror: 529 Tower Creation episodes. Original release: ~551 demo dirs.
   Small difference = LeRobot conversion dropping degenerate/incomplete demos.
   No timestep-by-timestep alignment attempted (per spec); the original release
   is the SOLE source once Part 3 proceeds.


### 1.4  Stop-condition evaluation

In [7]:
g = f["data"][SUBST[0][0]][SUBST[0][1]]
et = g["robot_observation/eef_times"][:]
ut = g["user_control/user_control_times"][:]
dt = np.diff(et)
print("streams share a clock (eef_times == user_control_times):", np.allclose(et, ut))
print("native sample dt: mean %.4fs (~%d Hz), asynchronous  min %.4f max %.4f"
      % (dt.mean(), round(1 / dt.mean()), dt.min(), dt.max()))
print("the mini file is NOT resampled; the full file is named '..._aligned_dataset.hdf5'.")

state_ok       = True    # eef_poses + joint_states_arm verified non-zero
action_ok      = False   # no trainable action in-file; needs robosuite or reconstruction
images_in_file = False   # only external MP4 paths in demo attrs
regular_10fps  = False   # ~500 Hz asynchronous in the mini file

print("")
print("Part 1.4 checklist:")
print("  proprioceptive state present & non-zero ........", state_ok, "  (PASS)")
print("  trainable action present in-file ..............", action_ok, "  (BLOCK - decision needed)")
print("  images present in-file ........................", images_in_file, "  (BLOCK - MP4 pipeline)")
print("  regular ~10 fps grid .........................", regular_10fps, "  (BLOCK - async resample)")

PROCEED = bool(state_ok and action_ok and images_in_file)
print("")
print("PROCEED to Part 3 training this run? ", PROCEED)
print("The state stop-condition (empty/unusable state) is NOT triggered - state is fine.")
print("But three unanticipated schema facts require a deliberate, separately reviewed")
print("decision before training. Parts 3-4 stay inert until PROCEED is True.")

streams share a clock (eef_times == user_control_times): True
native sample dt: mean 0.0016s (~624 Hz), asynchronous  min 0.0014 max 0.0096
the mini file is NOT resampled; the full file is named '..._aligned_dataset.hdf5'.

Part 1.4 checklist:
  proprioceptive state present & non-zero ........ True   (PASS)
  trainable action present in-file .............. False   (BLOCK - decision needed)
  images present in-file ........................ False   (BLOCK - MP4 pipeline)
  regular ~10 fps grid ......................... False   (BLOCK - async resample)

PROCEED to Part 3 training this run?  False
The state stop-condition (empty/unusable state) is NOT triggered - state is fine.
But three unanticipated schema facts require a deliberate, separately reviewed
decision before training. Parts 3-4 stay inert until PROCEED is True.


## Part 2 - The redesigned phase model  *(runs on the mini set)*

Uses the only gripper signal in this source, `user_control[:, 20]` (grasp bool), with the
same open/close-transition logic as `scp/phases.py`.

In [8]:
def grasp_bool(demo_grp):
    return demo_grp["user_control/user_control"][:, 20].astype(int)

def transitions(gb):
    return np.flatnonzero(np.diff(gb) != 0)

# 2.1  filter degenerate episodes
rows = []
for u, dk in demos:
    gb = grasp_bool(f["data"][u][dk])
    ch = transitions(gb)
    rows.append(dict(user=u, demo=dk, T=len(gb), n_trans=len(ch),
                     n_close=int((np.diff(gb) > 0).sum()),
                     n_open=int((np.diff(gb) < 0).sum())))
R = pd.DataFrame(rows)
R["degenerate_len1"] = R["T"] == 1
R["short_lt10"]      = R["T"] < 10
R["zero_grip_trans"] = R["n_trans"] == 0
drop = R["degenerate_len1"] | R["short_lt10"] | R["zero_grip_trans"]
print("mini-set filter (counts scale up on the full set):")
print("  length == 1 ............", int(R["degenerate_len1"].sum()))
print("  length < 10 ...........", int(R["short_lt10"].sum()))
print("  zero grasp transitions ", int(R["zero_grip_trans"].sum()))
print("  -> dropped %d / %d; usable = %d" % (int(drop.sum()), len(R), int((~drop).sum())))
USABLE = R[~drop].reset_index(drop=True)

mini-set filter (counts scale up on the full set):
  length == 1 ............ 0
  length < 10 ........... 0
  zero grasp transitions  2
  -> dropped 2 / 10; usable = 8


In [9]:
# 2.2 / 2.3 / 2.4  segment, K=2 pooling, empty-phase check
def phase_windows_K2(gb):
    return {"closed": gb == 1, "open": gb == 0}

seg_counts, cycle_counts, empty_any = [], [], 0
for _, r in USABLE.iterrows():
    gb = grasp_bool(f["data"][r["user"]][r["demo"]])
    w = phase_windows_K2(gb)
    seg_counts.append(len(transitions(gb)) + 1)
    cycle_counts.append(int((np.diff(gb) > 0).sum()))
    if not w["closed"].any() or not w["open"].any():
        empty_any += 1
seg_counts = np.array(seg_counts); cycle_counts = np.array(cycle_counts)
print("after the 2.1 filter, on the mini set:")
print("  segment count : min %d  median %.0f  max %d"
      % (seg_counts.min(), np.median(seg_counts), seg_counts.max()))
print("  grasp cycles  : min %d  median %.0f  max %d"
      % (cycle_counts.min(), np.median(cycle_counts), cycle_counts.max()))
print("  Option A (K=2) empty-phase rate : %d/%d = %.1f%%"
      % (empty_any, len(USABLE), 100 * empty_any / len(USABLE)))
for line in [
  "",
  "  Confirmed: with the 2.1 filter every usable episode has >=1 closed and >=1 open",
  "  window by construction, so K=2 has a 0% empty-phase rate. The multi-cycle structure",
  "  (median cycles ~ the LeRobot pass's 7) makes the LIBERO 4-tuple inapplicable; pooling",
  "  by category across all cycles is the same operation used for LIBERO.",
  "",
  "  NOTE contra Redesign doc 2.3: the original release DOES carry a continuous finger",
  "  width (joint_states_gripper[:,0], 0..0.04). So Option B (K=4 with finger-width",
  "  settling, exactly scp/phases.py) IS portable here, unlike the LeRobot mirror.",
  "  Left as the documented secondary option, not the default.",
]:
    print(line)

after the 2.1 filter, on the mini set:
  segment count : min 2  median 8  max 29
  grasp cycles  : min 1  median 4  max 14
  Option A (K=2) empty-phase rate : 0/8 = 0.0%

  Confirmed: with the 2.1 filter every usable episode has >=1 closed and >=1 open
  window by construction, so K=2 has a 0% empty-phase rate. The multi-cycle structure
  (median cycles ~ the LeRobot pass's 7) makes the LIBERO 4-tuple inapplicable; pooling
  by category across all cycles is the same operation used for LIBERO.

  NOTE contra Redesign doc 2.3: the original release DOES carry a continuous finger
  width (joint_states_gripper[:,0], 0..0.04). So Option B (K=4 with finger-width
  settling, exactly scp/phases.py) IS portable here, unlike the LeRobot mirror.
  Left as the documented secondary option, not the default.


## Part A - Verify D3 before anything else

Obtain `SawyerTowerCreation_aligned_dataset.hdf5` from the full release **without** the
23 GB of videos (D2 needs none): parse the zip's central directory via HTTP range requests,
then range-GET and inflate just that one member (~507 MB compressed -> 1.12 GB).

In [10]:
import os, urllib.request, struct, zlib
ALIGNED = None
for p in ["SawyerTowerCreation_aligned_dataset.hdf5",
          "_tmp_roboturk/SawyerTowerCreation_aligned_dataset.hdf5",
          "/content/SawyerTowerCreation_aligned_dataset.hdf5"]:
    if os.path.exists(p):
        ALIGNED = p; break

if ALIGNED is None:
    URL = "http://downloads.cs.stanford.edu/downloads/roboturk_real_dataset/SawyerTowerCreation_dataset.zip"
    def rng(a, b):
        return urllib.request.urlopen(urllib.request.Request(URL, headers={"Range": f"bytes={a}-{b}"}), timeout=300).read()
    total = int(urllib.request.urlopen(urllib.request.Request(URL, method="HEAD"), timeout=60).headers["Content-Length"])
    tail = rng(total - 1_200_000, total - 1)
    k, rec = 0, None
    while (k := tail.find(b"PK\x01\x02", k)) >= 0:
        n = struct.unpack("<H", tail[k+28:k+30])[0]; m = struct.unpack("<H", tail[k+30:k+32])[0]
        nm = tail[k+46:k+46+n].decode("utf-8", "replace")
        if nm.endswith(".hdf5"):
            rec = dict(method=struct.unpack("<H", tail[k+10:k+12])[0],
                       csize=struct.unpack("<I", tail[k+20:k+24])[0],
                       usize=struct.unpack("<I", tail[k+24:k+28])[0],
                       lho=struct.unpack("<I", tail[k+42:k+46])[0],
                       extra=tail[k+46+n:k+46+n+m])
        k += 4
    usize, csize, lho = rec["usize"], rec["csize"], rec["lho"]
    if 0xFFFFFFFF in (usize, csize, lho):            # zip64
        e = rec["extra"]; i = 0
        while i + 4 <= len(e):
            hid, sz = struct.unpack("<HH", e[i:i+4]); i += 4
            if hid == 1:
                b = e[i:i+sz]; j = 0
                if usize == 0xFFFFFFFF: usize = struct.unpack("<Q", b[j:j+8])[0]; j += 8
                if csize == 0xFFFFFFFF: csize = struct.unpack("<Q", b[j:j+8])[0]; j += 8
                if lho == 0xFFFFFFFF:   lho = struct.unpack("<Q", b[j:j+8])[0]
            i += sz
    lfh = rng(lho, lho + 29)
    ds = lho + 30 + struct.unpack("<H", lfh[26:28])[0] + struct.unpack("<H", lfh[28:30])[0]
    d = zlib.decompressobj(-zlib.MAX_WBITS)
    ALIGNED = "SawyerTowerCreation_aligned_dataset.hdf5"
    pos, end, CH = ds, ds + csize - 1, 8 << 20
    with open(ALIGNED, "wb") as fo:
        while pos <= end:
            ch = rng(pos, min(pos + CH - 1, end)); pos += len(ch)
            fo.write(d.decompress(ch))
        fo.write(d.flush())
print("aligned HDF5:", ALIGNED, os.path.getsize(ALIGNED), "bytes")

aligned HDF5: SawyerTowerCreation_aligned_dataset.hdf5 1118640480 bytes


In [11]:
import h5py, numpy as np, json
f = h5py.File(ALIGNED, "r")
demos = [(u, dk) for u in f["data"].keys() for dk in f["data"][u].keys()]
print("data/ attrs:", dict(f["data"].attrs), " ->", len(demos), "demos,", len(f["data"]), "users")

# D3 check: are the shipped *_times a regular grid?  what does user_control[:,21] (UTC) say?
rng_np = np.random.default_rng(0)
cv_times, cv_utc, rate_utc = [], [], []
for u, dk in [demos[i] for i in rng_np.choice(len(demos), 15, replace=False)]:
    g = f["data"][u][dk]
    tt = g["user_control/user_control_times"][:]
    utc = g["user_control/user_control"][:, 21]
    if len(tt) < 5: continue
    d1 = np.diff(tt); d2 = np.diff(utc); d2 = d2[d2 > 0]
    cv_times.append(d1.std() / d1.mean())
    cv_utc.append(d2.std() / d2.mean()); rate_utc.append(1 / np.median(d2))
    print("  %-9s T=%5d  *_times dt=%.4fs cv=%.2f | UTC[:,21] dt_med=%.4fs (~%.1fHz) cv=%.2f"
          % (dk, len(tt), d1.mean(), cv_times[-1], np.median(d2), 1/np.median(d2), cv_utc[-1]))
scale = np.polyfit(f["data"][demos[0][0]][demos[0][1]]["user_control/user_control"][:, 21],
                   f["data"][demos[0][0]][demos[0][1]]["user_control/user_control_times"][:], 1)[0]
print()
print("shipped *_times vs UTC scale factor : %.5f  (~ UTC / %.1f) -> *_times is a miscalibrated clock"
      % (scale, 1/scale))
print("UTC clock: median rate ~%.1f Hz, but CV ~%.2f with pause-gaps -> NOT a clean fixed grid"
      % (np.median(rate_utc), np.median(cv_utc)))
print("FINDING (D3): NOT aligned-as-shipped in time -> resample to a fixed 20 Hz grid on the UTC clock.")

data/ attrs: {'total_demos': np.int64(551), 'total_time': np.float64(137904.80889582634)}  -> 551 demos, 43 users
  demo_3    T= 3087  *_times dt=0.0011s cv=0.35 | UTC[:,21] dt_med=0.0500s (~20.0Hz) cv=0.23
  demo_14   T= 4326  *_times dt=0.0016s cv=0.27 | UTC[:,21] dt_med=0.0501s (~20.0Hz) cv=1.22
  demo_13   T= 3472  *_times dt=0.0016s cv=0.33 | UTC[:,21] dt_med=0.0501s (~20.0Hz) cv=4.90
  demo_4    T= 2597  *_times dt=0.0016s cv=0.20 | UTC[:,21] dt_med=0.0502s (~19.9Hz) cv=1.72
  demo_14   T= 4582  *_times dt=0.0016s cv=1.44 | UTC[:,21] dt_med=0.0501s (~20.0Hz) cv=2.20
  demo_15   T=  221  *_times dt=0.0027s cv=0.17 | UTC[:,21] dt_med=0.0495s (~20.2Hz) cv=1.71
  demo_26   T= 4516  *_times dt=0.0017s cv=1.43 | UTC[:,21] dt_med=0.0501s (~19.9Hz) cv=4.29
  demo_8    T= 4095  *_times dt=0.0016s cv=0.35 | UTC[:,21] dt_med=0.0501s (~20.0Hz) cv=1.97
  demo_21   T= 5075  *_times dt=0.0017s cv=1.33 | UTC[:,21] dt_med=0.0501s (~20.0Hz) cv=4.16
  demo_20   T= 1331  *_times dt=0.0017s cv=0.28 |

## Part B - Part 2's filtering & segmentation on the full release (531 demos)

In [12]:
rows = []
for u, dk in demos:
    gb = (f["data"][u][dk]["user_control/user_control"][:, 20] > 0.5).astype(int)
    ch = np.flatnonzero(np.diff(gb) != 0)
    rows.append(dict(user=u, demo=dk, T=len(gb), n_trans=len(ch),
                     has_closed=bool((gb == 1).any()), has_open=bool((gb == 0).any())))
T = np.array([r["T"] for r in rows]); NT = np.array([r["n_trans"] for r in rows])
drop = (T == 1) | (T < 10) | (NT == 0)
usable = [r for r, dd in zip(rows, drop) if not dd]
print("full release: %d demos" % len(rows))
print("  length == 1 ............ %d" % int((T == 1).sum()))
print("  length < 10 ........... %d" % int((T < 10).sum()))
print("  zero grasp transitions  %d" % int((NT == 0).sum()))
print("  -> dropped %d ; USABLE = %d" % (int(drop.sum()), len(usable)))

seg, cyc = [], []
for r in usable:
    gb = (f["data"][r["user"]][r["demo"]]["user_control/user_control"][:, 20] > 0.5).astype(int)
    seg.append((np.diff(gb) != 0).sum() + 1)
    cyc.append(int((np.diff(gb) > 0).sum()))
seg, cyc = np.array(seg), np.array(cyc)
true_empty = sum(1 for r in usable if not r["has_closed"] or not r["has_open"])
def q(a):
    return dict(min=int(a.min()), p10=round(float(np.percentile(a, 10)), 1),
               p25=round(float(np.percentile(a, 25)), 1), median=round(float(np.median(a)), 1),
               p75=round(float(np.percentile(a, 75)), 1), p90=round(float(np.percentile(a, 90)), 1),
               max=int(a.max()), mean=round(float(a.mean()), 1))
print("segment-count dist :", q(seg))
print("cycle-count   dist :", q(cyc))
print("K=2 pooled-window empty-phase rate after filter : %d/%d = %.2f%%   %s"
      % (true_empty, len(usable), 100*true_empty/len(usable),
         "(0% as expected)" if true_empty == 0 else "FLAG - report as-is"))
USABLE_KEYS = [(r["user"], r["demo"]) for r in usable]

full release: 551 demos
  length == 1 ............ 0
  length < 10 ........... 0
  zero grasp transitions  20
  -> dropped 20 ; USABLE = 531


segment-count dist : {'min': 2, 'p10': 7.0, 'p25': 11.0, 'median': 16.0, 'p75': 21.0, 'p90': 27.0, 'max': 46, 'mean': 17.0}
cycle-count   dist : {'min': 1, 'p10': 3.0, 'p25': 5.0, 'median': 8.0, 'p75': 10.0, 'p90': 13.0, 'max': 22, 'mean': 8.2}
K=2 pooled-window empty-phase rate after filter : 0/531 = 0.00%   (0% as expected)


## Part C - state-only BC-RNN, updated for D1-D3

`PROCEED = True`. No image code path (D2). Target is the D1 native action on the D3 20 Hz grid.

In [13]:
PROCEED = True

### C.1  Data pipeline - resample to 20 Hz and cache

In [14]:
CACHE = "roboturk_20hz.npz"
HZ, GRIP = 20.0, 7   # action col 7 = grasp bool

def _lin(t, v, gt):     return np.stack([np.interp(gt, t, v[:, j]) for j in range(v.shape[1])], 1)
def _near(t, v, gt):
    j = np.searchsorted(t, gt).clip(1, len(t) - 1)
    left = np.abs(gt - t[j - 1]) <= np.abs(t[j] - gt)
    return v[np.where(left, j - 1, j)]

if os.path.exists(CACHE):
    print("using cached", CACHE)
else:
    S_, A_, E_, Tlist = [], [], [], []
    for i, (u, dk) in enumerate(USABLE_KEYS):
        g = f["data"][u][dk]
        utc = g["user_control/user_control"][:, 21].astype(np.float64)
        keep = np.concatenate([[True], np.diff(utc) > 0]); utc = utc[keep]
        uc  = g["user_control/user_control"][:][keep]
        eef = g["robot_observation/eef_poses"][:][keep]
        arm = g["robot_observation/joint_states_arm"][:][keep]
        grp = g["robot_observation/joint_states_gripper"][:][keep]
        if len(utc) < 20: continue
        grid = np.arange(utc[0], utc[-1], 1.0 / HZ)
        if len(grid) < 32: continue
        st  = np.concatenate([_lin(utc, eef, grid), _lin(utc, arm, grid), _lin(utc, grp, grid)], 1)
        act = np.concatenate([_lin(utc, uc[:, 0:7], grid), _near(utc, uc[:, 20:21], grid)], 1)
        S_.append(st.astype(np.float32)); A_.append(act.astype(np.float32))
        E_.append(np.full(len(grid), len(Tlist), np.int32)); Tlist.append(len(grid))
    np.savez_compressed(CACHE, S=np.concatenate(S_), A=np.concatenate(A_),
                        E=np.concatenate(E_), ep_T=np.array(Tlist))
    print("cached", CACHE)

d = np.load(CACHE)
S, A, E, ep_T = d["S"], d["A"], d["E"], d["ep_T"]
n_ep = len(ep_T)
print("episodes %d | timesteps %d | state %d | action %d" % (n_ep, len(S), S.shape[1], A.shape[1]))
print("episode length (20Hz): median %d  p25 %d  p75 %d" %
      (np.median(ep_T), np.percentile(ep_T, 25), np.percentile(ep_T, 75)))

cached roboturk_20hz.npz


episodes 531 | timesteps 2652847 | state 37 | action 8
episode length (20Hz): median 5885  p25 4627  p75 5981


### C.2  Splits - policy train/val kept physically separate from Part D calib/eval

In [15]:
rng = np.random.default_rng(0)
order = rng.permutation(n_ep)
n_hold = int(round(0.30 * n_ep))
calib_eval_ep = np.sort(order[:n_hold])          # Part D ONLY
trainval_ep   = np.sort(order[n_hold:])
k = int(round(0.85 * len(trainval_ep)))
train_ep, val_ep = np.sort(trainval_ep[:k]), np.sort(trainval_ep[k:])
n_cal = len(calib_eval_ep) // 2
cal_ep, eval_ep = calib_eval_ep[:n_cal], calib_eval_ep[n_cal:]
print("policy: train %d  val %d   |   Part D: cal %d  eval %d  (never trained on)"
      % (len(train_ep), len(val_ep), len(cal_ep), len(eval_ep)))

def ep_slice(i):
    m = E == i
    return S[m], A[m]

tr  = np.concatenate([ep_slice(i)[0] for i in train_ep])
s_mu, s_sd = tr.mean(0), tr.std(0); s_sd[s_sd < 1e-6] = 1.0
tra = np.concatenate([ep_slice(i)[1] for i in train_ep])
a_mu, a_sd = tra.mean(0), tra.std(0); a_sd[a_sd < 1e-6] = 1.0
a_mu[GRIP], a_sd[GRIP] = 0.0, 1.0               # keep grasp bool in {0,1}
norm_s  = lambda x: (x - s_mu) / s_sd
norm_a  = lambda x: (x - a_mu) / a_sd
denorm_a = lambda x: x * a_sd + a_mu

policy: train 316  val 56   |   Part D: cal 79  eval 80  (never trained on)


### C.3  Architecture - state-only BC-RNN + GMM head (no vision layers)

In [16]:
import torch, torch.nn as nn, math
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

class BCRNN(nn.Module):
    def __init__(self, sdim, adim, hidden=256, layers=2, modes=5):
        super().__init__()
        self.modes, self.adim = modes, adim
        self.enc = nn.Sequential(nn.Linear(sdim, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU())
        self.rnn = nn.LSTM(256, hidden, layers, batch_first=True)
        self.head = nn.Linear(hidden, modes * (2 * adim + 1))
    def forward(self, s, hx=None):
        B, T, _ = s.shape
        z = self.enc(s.reshape(B * T, -1)).reshape(B, T, -1)
        o, hx = self.rnn(z, hx)
        p = self.head(o).reshape(B, T, self.modes, 2 * self.adim + 1)
        return p[..., :self.adim], p[..., self.adim:2*self.adim].clamp(-5, 2), p[..., -1], hx

def gmm_nll(mu, logstd, logit, y):
    y = y.unsqueeze(-2)
    ll = (-0.5 * (((y - mu) / logstd.exp())**2 + 2*logstd + math.log(2*math.pi))).sum(-1)
    return -torch.logsumexp(torch.log_softmax(logit, -1) + ll, -1).mean()

def gmm_mean(mu, logit):
    return (torch.softmax(logit, -1).unsqueeze(-1) * mu).sum(-2)
print("state-only BC-RNN defined; device =", DEV)

state-only BC-RNN defined; device = cuda


### C.4  Training - bounded, checkpointed; log train/val loss

In [17]:
L, STRIDE, BS, EPOCHS, LR = 64, 32, 128, 15, 3e-4
def windows(ep_ids):
    xs, ys = [], []
    for i in ep_ids:
        s, a = ep_slice(i)
        s, a = norm_s(s).astype(np.float32), norm_a(a).astype(np.float32)
        for st in range(0, max(1, len(s) - L), STRIDE):
            ss, aa = s[st:st+L], a[st:st+L]
            if len(ss) < L:
                pad = L - len(ss)
                ss = np.concatenate([ss, np.zeros((pad, ss.shape[1]), np.float32)])
                aa = np.concatenate([aa, np.zeros((pad, aa.shape[1]), np.float32)])
            xs.append(ss); ys.append(aa)
    return torch.tensor(np.stack(xs)), torch.tensor(np.stack(ys))

Xtr, Ytr = windows(train_ep); Xva, Yva = windows(val_ep)
print("train windows %d  val windows %d" % (len(Xtr), len(Xva)))

model = BCRNN(S.shape[1], A.shape[1]).to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=LR)
CKPT, hist = "roboturk_bcrnn_state_only.pt", []
for ep in range(1, EPOCHS + 1):
    model.train(); perm = torch.randperm(len(Xtr)); tl = 0.0
    for b in range(0, len(Xtr), BS):
        idx = perm[b:b+BS]; xb, yb = Xtr[idx].to(DEV), Ytr[idx].to(DEV)
        mu, ls, lg, _ = model(xb); loss = gmm_nll(mu, ls, lg, yb)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tl += loss.item() * len(idx)
    model.eval(); vl = 0.0
    with torch.no_grad():
        for b in range(0, len(Xva), BS):
            xb, yb = Xva[b:b+BS].to(DEV), Yva[b:b+BS].to(DEV)
            vl += gmm_nll(*model(xb)[:3], yb).item() * len(xb)
    tl, vl = tl / len(Xtr), vl / len(Xva); hist.append((ep, tl, vl))
    torch.save({"model": model.state_dict(), "epoch": ep,
                "s_mu": s_mu, "s_sd": s_sd, "a_mu": a_mu, "a_sd": a_sd}, CKPT)
    print("epoch %2d  train_nll %8.3f  val_nll %8.3f" % (ep, tl, vl))

train windows 48415  val windows 8739


epoch  1  train_nll   -0.431  val_nll   -6.211


epoch  2  train_nll   -6.514  val_nll   -7.830


epoch  3  train_nll   -7.961  val_nll   -9.135


epoch  4  train_nll   -9.002  val_nll   -9.971


epoch  5  train_nll   -9.867  val_nll  -10.355


... (6 outputs left out)

epoch 12  train_nll  -12.000  val_nll  -11.786


epoch 13  train_nll  -11.850  val_nll  -12.321


epoch 14  train_nll  -12.481  val_nll  -12.656


epoch 15  train_nll  -12.533  val_nll  -13.017


### C.5  Checkpoint + fingerprint (D1-D3 recorded in the config)

In [18]:
import hashlib, sys, platform
sha = hashlib.sha256(open(CKPT, "rb").read()).hexdigest()
fingerprint = {
    "checkpoint_sha256": sha,
    "libs": {"python": sys.version.split()[0], "torch": torch.__version__,
             "numpy": np.__version__, "platform": platform.platform(),
             "device": torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu"},
    "train_config": {"arch": "state-only BC-RNN: MLP(37->256->256) + LSTM(256x2) + GMM(5 modes, 8-dim)",
                     "epochs": EPOCHS, "lr": LR, "batch": BS, "window": L, "stride": STRIDE,
                     "train_episodes": int(len(train_ep)), "val_episodes": int(len(val_ep)),
                     "final_train_nll": hist[-1][1], "final_val_nll": hist[-1][2]},
    "decisions": {
        "D1": "action = user_control[:,0:7] (IK joint targets) + user_control[:,20] (grasp bool); "
              "native fields, no reconstruction, no robosuite. JOINT-TARGET space - different from "
              "the LeRobot mirror (delta-EEF) and the sim side (delta-pos OSC); no cross-source comparison.",
        "D2": "state-only: eef_poses(7)+joint_states_arm(27)+joint_states_gripper(3) = 37-dim. No images.",
        "D3": "aligned file *_times are a miscalibrated clock (~1/40.7 scale); real clock is "
              "user_control[:,21] (UTC), ~20 Hz but jittered/gappy -> fixed 20 Hz grid, linear for "
              "continuous fields, nearest for the grasp bool.",
    },
    "phase_model": "Part 2 Option A, K=2 (closed/open) via user_control[:,20]",
}
json.dump(fingerprint, open("fingerprint.json", "w"), indent=2)
print(json.dumps(fingerprint, indent=2))
print("\nCheckpoint is frozen once fingerprinted - no further training against this hash.")

{
  "checkpoint_sha256": "a3f61533104d494368c5a055b324f8d31522b4623bb978b4290a7ada845446b0",
  "libs": {
    "python": "3.13.2",
    "torch": "2.6.0+cu118",
    "numpy": "2.2.3",
    "platform": "Windows-11-10.0.26200-SP0",
    "device": "NVIDIA GeForce RTX 3050 6GB Laptop GPU"
  },
  "train_config": {
    "arch": "state-only BC-RNN: MLP(37->256->256) + LSTM(256x2) + GMM(5 modes, 8-dim)",
    "epochs": 15,
    "lr": 0.0003,
    "batch": 128,
    "window": 64,
    "stride": 32,
    "train_episodes": 316,
    "val_episodes": 56,
    "final_train_nll": -12.532955588731074,
    "final_val_nll": -13.017185264108983
  },
  "decisions": {
    "D1": "action = user_control[:,0:7] (IK joint targets) + user_control[:,20] (grasp bool); native fields, no reconstruction, no robosuite. JOINT-TARGET space - different from the LeRobot mirror (delta-EEF) and the sim side (delta-pos OSC); no cross-source comparison.",
    "D2": "state-only: eef_poses(7)+joint_states_arm(27)+joint_states_gripper(3) = 37-d

## Part D - predicted-vs-demonstrated calibration

`s(t) = ‖a_pred(t) - a_demo(t)‖₂` on held-out episodes, both in the D1 **joint-target
action space**. **This is not the simulation side's commanded-vs-executed score** and is
not reported as the same quantity. K = 2 pooled categories (`closed`/`open`), `α = 0.10`,
equal split `α_k = 0.05`; `scp/calibrate.py` order-statistic threshold, finite-threshold
bound `n ≥ 19` (LIBERO milestones used `n ≥ 79`). All demos treated as "successes" - no
failure labels exist. Calibration-validity check only; **no detection / escalation claim.**

In [19]:
def min_calibration_size(alpha): return int(math.ceil(1.0 / alpha - 1.0))
def conformal_threshold(scores, alpha):
    s = np.sort(np.asarray([x for x in scores if np.isfinite(x)], float)); n = s.size
    if n == 0: return float("inf"), n, 0, False
    rank = int(math.ceil((1 - alpha) * (n + 1)))
    if rank > n: return float("inf"), n, rank, False
    return float(s[rank - 1]), n, rank, True

@torch.no_grad()
def episode_scores(i):
    s, a = ep_slice(i)
    mu, ls, lg, _ = model(torch.tensor(norm_s(s)[None].astype(np.float32)).to(DEV))
    ahat = denorm_a(gmm_mean(mu, lg)[0].cpu().numpy())
    err = np.linalg.norm(ahat - a, axis=1)
    return err, a[:, GRIP] > 0.5

AGG = np.max     # matches scp/scoring.py DEFAULT_AGG
def per_cat(ep_ids):
    o = {"closed": [], "open": []}
    for i in ep_ids:
        err, closed = episode_scores(i)
        if closed.any():    o["closed"].append(float(AGG(err[closed])))
        if (~closed).any(): o["open"].append(float(AGG(err[~closed])))
    return o

ALPHA, K = 0.10, 2; a_k = ALPHA / K
cal, ev = per_cat(cal_ep), per_cat(eval_ep)
print("alpha=%.2f K=%d alpha_k=%.3f  n_min(calibrate.py)=%d  (LIBERO used n>=79)"
      % (ALPHA, K, a_k, min_calibration_size(a_k)))
res = {}
for c in ("closed", "open"):
    q_, n, rank, feas = conformal_threshold(cal[c], a_k)
    cov = float(np.mean(np.asarray(ev[c]) <= q_)) if np.isfinite(q_) else 1.0
    res[c] = dict(q=q_, n_cal=n, rank=rank, feasible=feas, n_eval=len(ev[c]), coverage=cov)
    print("  %-6s n_cal=%3d rank=%3d feasible=%s  q_hat=%.4f  eval coverage=%.3f (target %.2f, n_eval=%d)"
          % (c, n, rank, feas, q_, cov, 1 - ALPHA, len(ev[c])))
joint = float(np.mean([
    ((not cl.any()) or np.max(er[cl]) <= res["closed"]["q"]) and
    ((not (~cl).any()) or np.max(er[~cl]) <= res["open"]["q"])
    for i in eval_ep for er, cl in [episode_scores(i)]]))
print("  JOINT (both categories within threshold) eval coverage = %.3f  (target >= %.2f)" % (joint, 1 - ALPHA))

json.dump({"alpha": ALPHA, "K": K, "alpha_k": a_k,
           "n_min_calibrate_py": min_calibration_size(a_k), "n_min_libero": 79,
           "per_category": {k: {kk: (None if not np.isfinite(vv) else vv) for kk, vv in v.items()}
                            for k, v in res.items()},
           "joint_coverage_eval": joint,
           "score": "predicted-vs-demonstrated ||a_pred - a_demo||_2 in JOINT-TARGET space; "
                    "NOT the sim commanded-vs-executed score",
           "caveats": "calibration-validity only; all demos = successes (no failure labels); "
                      "no detection/escalation claim"},
          open("partD.json", "w"), indent=2)
print("wrote partD.json")

alpha=0.10 K=2 alpha_k=0.050  n_min(calibrate.py)=19  (LIBERO used n>=79)
  closed n_cal= 79 rank= 76 feasible=True  q_hat=2.2836  eval coverage=1.000 (target 0.90, n_eval=80)
  open   n_cal= 79 rank= 76 feasible=True  q_hat=2.2111  eval coverage=1.000 (target 0.90, n_eval=80)


  JOINT (both categories within threshold) eval coverage = 1.000  (target >= 0.90)
wrote partD.json


### Part D result

The `n ≥ 19` feasibility bound is now cleared comfortably (the full release gives ~80
held-out episodes per category vs the mini set's ≤ 8). Thresholds, per-category and joint
coverage on the eval slice are printed above and saved to `partD.json`. Coverage at/above
the nominal `1 − α = 0.90` on nominal demonstrations confirms the split-conformal machinery
is wired correctly for this predicted-vs-demonstrated score; it makes **no** claim about
detecting real failures or OOD behaviour (neither is present here).

## Out of scope (unchanged): ACT training; live arbitration; genuine failure detection.